# RAG-Augmented Multi-Agent Research Pipeline

This notebook demonstrates a **Retrieval-Augmented Generation (RAG)** workflow combined with a **multi-agent orchestration** system (CrewAI) to answer scientific research questions grounded in uploaded papers.

### Research Question
> *Do more abstract cognitive tasks preferentially activate more anterior regions of the prefrontal cortex (PFC)?*

### Papers in the Vector Database
| ID | Citation | Role |
|---|---|---|
| `2007_badre.pdf` | Badre & D'Esposito (2007), *J. Cogn. Neurosci.* | Defines abstract vs. concrete task hierarchy |
| `2023_multitask.pdf` | Ito et al. (2023), *Nat. Neurosci.* | 26-task fMRI dataset (MDTB) |
| `2024_demand.pdf` | Assem et al. (2024), *Cereb. Cortex* | Multiple-demand system, 3 executive tasks |

### Pipeline Architecture
```
[PDF Corpus] → [FAISS Vector DB] → [RAG Tool]
                                         ↓
              Agent 1: Task Classifier (retrieves definitions + task lists)
                                         ↓
              Agent 2: Activation Analyst (retrieves PFC activation data per task)
                                         ↓
              Agent 3: Scientific Verifier (cross-checks claims, searches bioRxiv)
```

## 1. Environment Setup

In [1]:
from dotenv import dotenv_values
import os
import sys
import warnings

# HPC-specific paths for Arrow and FAISS — adjust or remove for local environments
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/24.0.0/lib/python3.12/site-packages')
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/CUDA/gcc12/cuda12.6/faiss/1.12.0/lib/python3.12/site-packages')

warnings.filterwarnings('ignore')

config = dotenv_values(".env")
os.environ["OPENROUTER_API_KEY"] = config['OPENROUTER_API_KEY']
os.environ["HUGGINGFACEHUB_API_TOKEN"] = config['HF_API_KEY']
OPENROUTER_API_KEY = config['OPENROUTER_API_KEY']

print("Environment loaded.")

Environment loaded.


## 2. LLM Configuration

We use **`nvidia/nemotron-3-nano`** via OpenRouter — a free model with reliable instruction-following for structured outputs.

> **Note on model selection:** Free-tier models have variable tool-calling reliability. If this model hits rate limits or produces malformed tool calls, swap to `meta-llama/llama-4-maverick:free` or `qwen/qwen3-235b-a22b:free` — both handle structured JSON output well.

In [2]:
from crewai import Agent, Task, Crew, LLM

llm = LLM(
    model="openrouter/nvidia/nemotron-3-nano-30b-a3b:free",
    # model="openrouter/meta-llama/llama-4-maverick:free",
    api_key=config['OPENROUTER_API_KEY'],
    base_url="https://openrouter.ai/api/v1",
    # Fallback options if rate-limited:
    # model="openrouter/meta-llama/llama-4-maverick:free"
    # model="openrouter/qwen/qwen3-235b-a22b:free"
)

print(f"LLM configured: {llm.model}")

LLM configured: nvidia/nemotron-3-nano-30b-a3b:free


## 3. Vector Database (RAG Component)

We build a **FAISS** index over the three PDFs using `all-MiniLM-L6-v2` embeddings — a lightweight but effective sentence transformer.

**Chunking strategy:**
- `chunk_size=1500` characters — large enough to preserve argument context across sentences
- `chunk_overlap=200` — prevents losing information at chunk boundaries
- Retriever returns top `k=6` chunks per query

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

PDF_PATHS = [
    'data/2007_badre.pdf',      # Badre & D'Esposito — hierarchical PFC framework
    'data/2023_multitask.pdf',  # Ito et al. — MDTB 26-task fMRI dataset
    'data/2024_demand.pdf',     # Assem et al. — multiple-demand executive tasks
]

# Load and split
docs = []
for pdf in PDF_PATHS:
    loaded = PyPDFLoader(pdf).load()
    docs.extend(loaded)
    print(f"  Loaded {len(loaded)} pages from {pdf}")

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
chunks = splitter.split_documents(docs)
print(f"\nTotal chunks: {len(chunks)}")

# Embed and index
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 6})

print("FAISS index built successfully.")

  Loaded 18 pages from data/2007_badre.pdf
  Loaded 28 pages from data/2023_multitask.pdf
  Loaded 19 pages from data/2024_demand.pdf

Total chunks: 258


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index built successfully.


## 4. Tools

### 4a. RAG Tool — `paper_search`

Queries the local FAISS index and returns the top-k chunks with source metadata. This is the primary retrieval mechanism for all three agents.

In [4]:
from crewai.tools import tool

retrieval_log = []   # global log — populated during crew.kickoff()

@tool("paper_search")
def paper_search(query: str) -> str:
    """
    Search the uploaded research papers for passages relevant to the query.
    ...
    """
    retrieved_docs = retriever.invoke(query)
    if not retrieved_docs:
        return "No relevant passages found. Try rephrasing your query."

    result_text = "\n\n---\n\n".join([
        f"[Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in retrieved_docs
    ])

    # ── Log for evaluation ──────────────────────────────────────────
    retrieval_log.append({
        "query":    query,
        "chunks":   [
            {
                "source": doc.metadata.get("source"),
                "page":   doc.metadata.get("page"),
                "text":   doc.page_content[:300],
            }
            for doc in retrieved_docs
        ],
    })
    # ───────────────────────────────────────────────────────────────

    return result_text

### 4b. Web Tool — `biorxiv_search`

Scrapes bioRxiv search results to check whether the findings match recent preprint literature. Includes error handling and graceful fallback.

In [5]:
import requests
from bs4 import BeautifulSoup

@tool("biorxiv_search")
def biorxiv_search(query: str) -> str:
    """
    Search bioRxiv for preprints relevant to the query.
    Use this to check whether findings in the local papers are corroborated 
    by recent neuroscience preprints. Returns titles and abstracts of up to 
    5 results. If no results are found or the search fails, returns a specific 
    explanation so the agent can reason about the absence of evidence.
    """
    try:
        url = f"https://www.biorxiv.org/search/{query.replace(' ', '%20')}"
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        results = []
        for article in soup.select(".highwire-article-citation")[:5]:
            title = article.select_one(".highwire-cite-title")
            abstract = article.select_one(".highwire-cite-snippet")
            if title:
                results.append(
                    f"Title: {title.get_text(strip=True)}\n"
                    f"Snippet: {abstract.get_text(strip=True) if abstract else 'No abstract available'}"
                )

        if results:
            return f"Found {len(results)} result(s) on bioRxiv:\n\n" + "\n\n".join(results)
        else:
            return (
                f"No preprints found on bioRxiv for query: '{query}'. "
                "This may indicate the topic is primarily covered in peer-reviewed journals, "
                "or that the query terms need adjustment. Consider rephrasing or checking "
                "PubMed for published literature."
            )

    except requests.exceptions.Timeout:
        return "bioRxiv search timed out. The site may be temporarily unavailable."
    except requests.exceptions.RequestException as e:
        return f"bioRxiv search failed with network error: {str(e)}. Proceed using local paper evidence only."
    except Exception as e:
        return f"Unexpected error during bioRxiv search: {str(e)}."

## 5. Agent Definitions

Each agent has a distinct scientific role. **Roles** describe professional identity; **goals** describe measurable success; **backstories** provide domain context that shapes how the LLM reasons.

In [6]:
# ─────────────────────────────────────────────────────────────
# Agent 1: Task Classification Specialist
# Responsibility: Extract and classify tasks by abstraction level
# ─────────────────────────────────────────────────────────────
task_classification_agent = Agent(
    role="Cognitive Task Classification Specialist",
    goal=(
        "Produce a complete, evidence-grounded classification of every cognitive task "
        "in the Ito (2023) and Assem (2024) papers as either abstract or concrete, "
        "using the operational definition provided by Badre & D'Esposito (2007). "
        "Every classification must cite specific retrieved text as justification."
    ),
    backstory=(
        "You are a cognitive neuroscientist with expertise in task taxonomy and "
        "hierarchical theories of cognitive control. You are meticulous about "
        "distinguishing tasks that require abstract rule selection (e.g., task-switching, "
        "working memory updating) from those that operate on concrete stimulus-response "
        "mappings (e.g., simple motor responses, passive viewing). You never classify "
        "a task without first retrieving its description from the papers."
    ),
    tools=[paper_search],
    allow_delegation=False,
    llm=llm,
    verbose=True
)

# ─────────────────────────────────────────────────────────────
# Agent 2: Neuroimaging Data Analyst
# Responsibility: Extract PFC activation findings per task from the papers
# ─────────────────────────────────────────────────────────────
activation_analyst_agent = Agent(
    role="Neuroimaging Data Analyst",
    goal=(
        "For each task in the Ito (2023) and Assem (2024) papers, retrieve and report "
        "the specific PFC subregions (e.g., frontopolar, DLPFC, premotor cortex, PMd, "
        "pre-PMd, IFJ) that showed significant activation, citing page numbers or "
        "figure references. Do NOT infer activation patterns — only report what is "
        "explicitly stated in retrieved text."
    ),
    backstory=(
        "You are an fMRI data analyst specializing in frontal lobe functional organization. "
        "You are trained to distinguish between frontopolar cortex (BA10), dorsolateral PFC "
        "(DLPFC, BA46/9), inferior frontal junction (IFJ), premotor cortex (PMd/PMv), "
        "and primary motor cortex (M1) — and you know that these regions span anterior-to-posterior "
        "PFC. You are rigorous about not extrapolating beyond what the data shows: if a paper "
        "does not report task-specific PFC subregion activation, you report that as a data gap "
        "rather than inventing a result."
    ),
    tools=[paper_search],
    allow_delegation=False,
    llm=llm,
    verbose=True
)

# ─────────────────────────────────────────────────────────────
# Agent 3: Scientific Claims Verifier
# Responsibility: Cross-check findings against literature and flag unsupported claims
# ─────────────────────────────────────────────────────────────
scientific_verifier_agent = Agent(
    role="Scientific Claims Verifier",
    goal=(
        "Evaluate whether the evidence from Agents 1 and 2 is sufficient to support "
        "the claim that more abstract tasks activate more anterior PFC. "
        "Flag any gaps, inconsistencies, or unsupported inferences. "
        "Search bioRxiv for corroborating or contradicting recent literature. "
        "Deliver a calibrated verdict: supported / partially supported / not supported, "
        "with explicit reasoning."
    ),
    backstory=(
        "You are a scientific peer reviewer with expertise in meta-analysis and "
        "replication science in cognitive neuroscience. You have reviewed manuscripts "
        "on prefrontal cortex organization for journals including Nature Neuroscience "
        "and Cerebral Cortex. You are skeptical of overinterpretation and always "
        "distinguish between 'the paper shows X' and 'the available evidence suggests X.' "
        "You note when a dataset (e.g., MDTB) was not designed to test anterior-posterior "
        "gradients explicitly, and flag when conclusions go beyond what the data support."
    ),
    tools=[paper_search, biorxiv_search],
    allow_delegation=False,
    llm=llm,
    verbose=True
)

print("Agents defined.")

Agents defined.


## 6. Task Definitions

Tasks specify *what to do*, *what format to produce*, and *which agent is responsible*. Clear expected outputs constrain the LLM and make outputs programmatically parseable.

In [10]:
# ─────────────────────────────────────────────────────────────
# Task 1: Retrieve the abstract/concrete framework + classify all tasks
# ─────────────────────────────────────────────────────────────
classify_tasks = Task(
    description=(
        "Complete the following steps in order:\n\n"
        "**Step 1 — Retrieve the definition of abstract vs. concrete tasks.**\n"
        "Use paper_search with the query: 'Badre D\'Esposito hierarchical abstraction "
        "definition concrete abstract levels representation'. "
        "Extract the operational definition: what makes a task abstract vs. concrete "
        "according to the hierarchical framework (e.g., the level at which competing "
        "representations must be resolved).\n\n"
        "**Step 2 — Retrieve the task list from Ito et al. (2023).**\n"
        "Use paper_search with: 'MDTB 26 tasks list multitask fMRI Ito 2023'. "
        "Extract the full list of tasks from the MDTB dataset.\n\n"
        "**Step 3 — Retrieve the task list from Assem et al. (2024).**\n"
        "Use paper_search with: 'Assem 2024 executive tasks n-back switch stop signal'. "
        "Extract the tasks used and their easy/hard conditions.\n\n"
        "**Step 4 — Classify each task.**\n"
        "Apply the Badre & D'Esposito definition to classify each task as Abstract or "
        "Concrete. For each task, state which level of the hierarchy it engages "
        "(response, feature, dimension, or context level) and provide a one-sentence "
        "justification citing evidence from the retrieved text."
    ),
    expected_output=(
        "A markdown table with columns:\n"
        "| Task Name | Paper | Abstract / Concrete | Hierarchy Level | Justification (with text evidence) |\n"
        "Followed by a brief paragraph summarizing the Badre & D'Esposito definition used."
    ),
    agent=task_classification_agent
)

# ─────────────────────────────────────────────────────────────
# Task 2: Retrieve PFC activation data per task
# ─────────────────────────────────────────────────────────────
analyze_activation = Task(
    description=(
        "Retrieve PFC-related findings from Ito (2023) and Assem (2024) only. "
        "Do NOT use Badre (2007) for activation data.\n\n"
        "**Step 1 — Assem (2024) sub-areal frontal findings.**\n"
        "Search: 'Assem 2024 n-back dorsal frontal 8Ad anterior DMN'\n"
        "Search: 'Assem 2024 stop CON switch DAN frontal preference vertices'\n"
        "Search: 'Assem 2024 p9-46v anterior dorsal posterior ventral border'\n"
        "Extract any named frontal areas and which task preferentially activated them.\n\n"
        "**Step 2 — Ito (2023) frontal hierarchy descriptions.**\n"
        "Search: 'Ito 2023 frontal hierarchy representational alignment CPRO n-back'\n"
        "Search: 'Ito 2023 anterior frontal cortex task topography'\n"
        "Search: 'multitask intrinsic hierarchy frontal cortex high demand'\n"
        "Extract any text describing where in frontal cortex tasks are represented.\n\n"
        "**Step 3 — Be explicit about what these papers can and cannot say.**\n"
        "Neither paper was designed to test rostro-caudal abstraction gradients. "
        "State clearly for each paper: (a) what frontal activation language IS present, "
        "(b) what the paper's actual design goal was, and (c) whether the available "
        "text supports any anterior-posterior inference at all."
    ),
    expected_output=(
        "A markdown table with columns:\n"
        "| Task | Paper | Frontal Region Named | Anterior or Posterior | "
        "Evidence Quote / Figure Reference |\n\n"
        "Only populate rows where text explicitly names a frontal region. "
        "Leave no row as N/A without a one-sentence explanation of why.\n\n"
        "'Data Gaps' section must distinguish between:\n"
        "- **Retrieval gap**: text exists but wasn't found\n"
        "- **Design gap**: paper never reported this because it wasn't the study goal\n"
    ),
    agent=activation_analyst_agent
)

classify_tasks = Task(
    description=(
        "Complete the following steps in order:\n\n"
        "**Step 1 — Retrieve the definition of abstract vs. concrete tasks.**\n"
        "Use paper_search with: 'Badre D\\'Esposito hierarchical abstraction "
        "definition concrete abstract levels representation'.\n\n"
        "**Step 2 — Retrieve the task lists from Ito (2023) and Assem (2024).**\n"
        "Use paper_search with: 'MDTB 26 tasks list multitask fMRI Ito 2023'.\n"
        "Use paper_search with: 'Assem 2024 executive tasks n-back switch stop signal'.\n\n"
        "**Step 3 — Apply the hierarchy carefully. The four levels are MUTUALLY EXCLUSIVE:\n"
        "- RESPONSE level (concrete): task requires choosing among motor responses only. "
        "No rule selection, no feature comparison. Examples: button press, go/no-go, passive viewing, rest.\n"
        "- FEATURE level: task requires selecting among perceptual feature-to-response mappings. "
        "Example: Stroop (colour word vs ink colour), visual search (target feature among distractors).\n"
        "- DIMENSION level: task requires selecting which perceptual dimension is relevant. "
        "Example: n-back (track identity across time), mental rotation (spatial dimension).\n"
        "- CONTEXT level: task requires selecting among sets of task rules or contextual mappings. "
        "Example: CPRO (permuted rule sets), task switching (switch between rule sets), "
        "theory of mind (social context rules).\n\n"
        "IMPORTANT: Passive tasks (movie watching, action observation, object viewing, biological motion, "
        "motor imagery, interval timing) involve NO rule selection — they are CONCRETE / RESPONSE level.\n"
        "Do NOT default every task to context level. Each task must be assessed individually.\n\n"
        "**Step 4 — Classify each task**, providing a one-sentence justification that names "
        "the specific competition type (response/feature/dimension/context) from the retrieved text."
    ),
    expected_output=(
        "A markdown table with columns:\n"
        "| Task Name | Paper | Abstract / Concrete | Hierarchy Level | Justification |\n\n"
        "REQUIRED: At least 6 tasks must be classified as Concrete. "
        "If all tasks are Abstract, you have made an error — re-read the definitions."
    ),
    agent=task_classification_agent
)


# ─────────────────────────────────────────────────────────────
# Task 3: Verify the central claim and situate in literature
# ─────────────────────────────────────────────────────────────

FRONTAL_ANATOMY_REFERENCE = """
FRONTAL LOBE ANTERIOR-POSTERIOR REFERENCE (Glasser HCP MMP1.0 parcellation):

ANTERIOR PFC (rostral):
- Area 10 / BA10 / frontopolar cortex — most anterior
- Area 9 (dorsomedial PFC, 9m)
- Area 8Ad (anterior part of area 8, borders DMN) — anterior
- Area p9-46v anterior-dorsal border — anterior
- Area 47 / 47l (lateral orbitofrontal) — anterior-ventral

MID-LATERAL PFC:
- Area 46 / 9-46v / p9-46v — mid-lateral DLPFC
- Area 9-46d — mid-lateral DLPFC
- Area 8Av / 8C — mid-frontal

POSTERIOR/PREMOTOR PFC:
- Area 6 / 6a (dorsal premotor, DAN-associated) — posterior frontal
- Area 6r / IFJ (inferior frontal junction) — posterior-lateral
- Area 55b — posterior lateral PFC
- FOP5 (frontal operculum) — posterior-ventral

KEY MAPPINGS FROM ASSEM (2024):
- 3>1 n-back preferring vertices → overlap with 8Ad (ANTERIOR)
- stop>no stop preferring vertices → overlap with CON, area 46, 6r (MID to POSTERIOR)
- switch>no switch preferring vertices → overlap with DAN, area 6a (POSTERIOR)
- p9-46v anterior-dorsal border → stop activations (MID-ANTERIOR)
- p9-46v posterior-ventral border → switch activations (MID-POSTERIOR)
"""

verify_claim = Task(
    description=(
        "Evaluate whether findings from Ito (2023) and Assem (2024) support:\n"
        "'More abstract cognitive tasks activate more anterior PFC regions.'\n\n"
        "IMPORTANT: Use the anatomical reference below to classify any named "
        "frontal region as anterior, mid, or posterior — do NOT wait for the "
        "papers to use the word 'anterior' explicitly.\n\n"
        f"{FRONTAL_ANATOMY_REFERENCE}\n\n"
        "**Step 1 — Map reported regions onto anterior-posterior axis.**\n"
        "From Task 2 output, take every named frontal region and classify it "
        "using the reference above. Then cross-reference with Task 1 "
        "classifications (abstract vs. concrete).\n"
        "Key mappings to apply from Assem (2024):\n"
        "- 3>1 n-back → area 8Ad → ANTERIOR\n"
        "- stop>no stop → area 46, 6r → MID to POSTERIOR\n"
        "- switch>no switch → area 6a → POSTERIOR\n\n"
        "**Step 2 — Test the gradient claim.**\n"
        "Given that n-back is classified as Abstract (context level) and "
        "activates 8Ad (anterior), while switch activates 6a (posterior) — "
        "does this partial pattern support the claim?\n\n"
        "**Step 3 — Search literature.**\n"
        "Search paper_search: 'Assem 2024 8Ad anterior frontal n-back DMN'\n"
        "Search paper_search: 'Ito 2023 frontal hierarchy representational topography'\n\n"
        "**Step 4 — Verdict.**\n"
        "Choose: Supported / Partially Supported / Not Supported.\n"
        "Be specific: cite which region-task pairs drive the verdict."
    ),
    expected_output=(
        "1. **Region Classification Table**\n"
        "| Task | Paper | Region Named | A/P Classification | Abstract? | Gradient Consistent? |\n\n"
        "2. **Evidence For** — specific region-task pairs that fit abstract→anterior\n"
        "3. **Evidence Against** — mismatches\n"
        "4. **Data Gaps** — design gap vs retrieval gap, per paper\n"
        "5. **Verdict** with one-paragraph justification citing specific regions\n"
        "6. **Key References**"
    ),
    agent=scientific_verifier_agent
)


print("Tasks defined.")

Tasks defined.


## 7. Run the Crew

In [11]:
import os
import tempfile

# Point CrewAI's SQLite storage to local scratch instead of /project NFS
local_db_dir = os.path.join(tempfile.gettempdir(), "crewai_storage")
os.makedirs(local_db_dir, exist_ok=True)
os.environ["CREWAI_STORAGE_DIR"] = local_db_dir

In [12]:
crew = Crew(
    agents=[task_classification_agent, activation_analyst_agent, scientific_verifier_agent],
    tasks=[classify_tasks, analyze_activation, verify_claim],
    max_rpm=10,       # Respect free-tier rate limits
    verbose=True,
    memory=False      # No cross-session memory; all context passed via task outputs
)

In [13]:
result = crew.kickoff()

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.4                                                                                        │
│  Latest version:  1.14.6                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 47d2c546-d69a-402f-988a-9d1390126d7f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Complete the following steps in order:                                                                   │
│                                                                                                                 │
│  **Step 1 — Retrieve the definition of abstract vs. concrete tasks.**                                           │
│  Use paper_search with: 'Badre D\'Esposito hierarchical abstraction definition concrete abstract levels         │
│  representation'.                                                                                               │
│                                                                                                                 │
│  **Step 2 — Retrieve the task lists from Ito (2023) and Assem (2024).**                                         │
│  Use paper_search with: 'MDTB 26 tasks list multitask fMRI Ito 2023'.                                           │
│  Use paper_search with: 'Assem 2024 executive tasks n-back switch stop signal'.                                 │
│                                                                                                                 │
│  **Step 3 — Apply the hierarchy carefully. The four levels are MUTUALLY EXCLUSIVE:                              │
│  - RESPONSE level (concrete): task requires choosing among motor responses only. No rule selection, no feature  │
│  comparison. Examples: button press, go/no-go, passive viewing, rest.                                           │
│  - FEATURE level: task requires selecting among perceptual feature-to-response mappings. Example: Stroop        │
│  (colour word vs ink colour), visual search (target feature among distractors).                                 │
│  - DIMENSION level: task requires selecting which perceptual dimension is relevant. Example: n-back (track      │
│  identity across time), mental rotation (spatial dimension).                                                    │
│  - CONTEXT level: task requires selecting among sets of task rules or contextual mappings. Example: CPRO        │
│  (permuted rule sets), task switching (switch between rule sets), theory of mind (social context rules).        │
│                                                                                                                 │
│  IMPORTANT: Passive tasks (movie watching, action observation, object viewing, biological motion, motor         │
│  imagery, interval timing) involve NO rule selection — they are CONCRETE / RESPONSE level.                      │
│  Do NOT default every task to context level. Each task must be assessed individually.                           │
│                                                                                                                 │
│  **Step 4 — Classify each task**, providing a one-sentence justification that names the specific competition    │
│  type (response/feature/dimension/context) from the retrieved text.                                             │
│  ID: 6239efbd-e015-4ba1-a02a-900e9246041f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cognitive Task Classification Specialist                                                                │
│                                                                                                                 │
│  Task: Complete the following steps in order:                                                                   │
│                                                                                                                 │
│  **Step 1 — Retrieve the definition of abstract vs. concrete tasks.**                                           │
│  Use paper_search with: 'Badre D\'Esposito hierarchical abstraction definition concrete abstract levels         │
│  representation'.                                                                                               │
│                                                                                                                 │
│  **Step 2 — Retrieve the task lists from Ito (2023) and Assem (2024).**                                         │
│  Use paper_search with: 'MDTB 26 tasks list multitask fMRI Ito 2023'.                                           │
│  Use paper_search with: 'Assem 2024 executive tasks n-back switch stop signal'.                                 │
│                                                                                                                 │
│  **Step 3 — Apply the hierarchy carefully. The four levels are MUTUALLY EXCLUSIVE:                              │
│  - RESPONSE level (concrete): task requires choosing among motor responses only. No rule selection, no feature  │
│  comparison. Examples: button press, go/no-go, passive viewing, rest.                                           │
│  - FEATURE level: task requires selecting among perceptual feature-to-response mappings. Example: Stroop        │
│  (colour word vs ink colour), visual search (target feature among distractors).                                 │
│  - DIMENSION level: task requires selecting which perceptual dimension is relevant. Example: n-back (track      │
│  identity across time), mental rotation (spatial dimension).                                                    │
│  - CONTEXT level: task requires selecting among sets of task rules or contextual mappings. Example: CPRO        │
│  (permuted rule sets), task switching (switch between rule sets), theory of mind (social context rules).        │
│                                                                                                                 │
│  IMPORTANT: Passive tasks (movie watching, action observation, object viewing, biological motion, motor         │
│  imagery, interval timing) involve NO rule selection — they are CONCRETE / RESPONSE level.                      │
│  Do NOT default every task to context level. Each task must be assessed individually.                           │
│                                                                                                                 │
│  **Step 4 — Classify each task**, providing a one-sentence justification that names the specific competition    │
│  type (response/feature/dimension/context) from the retrieved text.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: paper_search                                                                                             │
│  Args: {'query': "Badre D'Esposito hierarchical abstraction definition concrete abstract levels                 │
│  representation"}                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#5) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: paper_search                                                                                             │
│  Output: [Source: data/2024_demand.pdf | Page: 15]                                                              │
│  16 | Cerebral Cortex, 2024, Vol. 34, No. 2                                                                     │
│  The n-back contrast was slightly right lateralized, showing                                                    │
│  strong activation throughout core MD ( Figs. 4 and 5)w i t h                                                   │
│  strongest activations shifted mainly toward core MD-coreMD,                                                    │
│  coreMD-DAN, or coreMD-DMN borders ( Fig. 7). N-back also                                                       │
│  showed the least deactivation of DMN, with the strongest effects                                               │
│  in inferior parietal, medial parietal, and dorsal frontal portions, a                                          │
│  canonicalDMNfractionationusuallyimplicatedinmemoryrecall                                                       │
│  and spatial imagery studies (                                                                                  │
│  Fig. 4 and Supplementary Fig. 2)                                                                               │
│  (Andrews-Hanna et al. 2010; Wen et al. 2020; Shao et al. 2023). A                                              │
│  recent study also demonstrated co-engagement of intermediate                                                   │
│  nodes between FPN and DMN in a 1-back>0-back task (Murphy                                                      │
│  et al. 2018). Murphy et al. suggested the interaction between                                                  │
│  FPN and DMN plays a role in recalling detailed information                                                     │
│  from the immediate past. We speculate that the engagement of                                                   │
│  the intersection between parietal DMN (e.g. PGs, part of 2020-                                                 │
│  penumbra) and core MD patches reflects engagement of the                                                       │
│  episodic recall network (                                                                                      │
│  Rugg and Vilberg 2013) though this needs                                                                       │
│  confirmation through a recall-focused paradigm investigated in                                                 │
│  a spatially precise approach similar to the current study.                                                     │
│  Switch>no switch activations were left lateralized, with                                                       │
│  strongest activations in core MD and DAN (Fig. 4c), in line with                                               │
│  previous results highlighting dorsal fronto-parietal activations                                               │
│  (Braver et al. 2003; Crone et al. 2006; Tsumura et al. 2021).                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  [Source: data/2007_badre.pdf | Page: 13]              

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scientific Claims Verifier                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **1. Region Classification Table**                                                                             │
│                                                                                                                 │
│  | Task | Paper | Region Named | A/P Classification | Abstract? | Gradient Consistent? |                        │
│  |------|-------|--------------|-------------------|-----------|----------------------|                         │
│  | n‑back | Assem 2024 | 8Ad (anterior part of area 8) | ANTERIOR | Yes | **Yes** |                             │
│  | stop | Assem 2024 | CON (area 46, 6r) | MID‑POSTERIOR (i.e., not anterior) | Yes | **No** |                  │
│  | switch | Assem 2024 | DAN / area 6a | POSTERIOR | Yes | **No** |                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **2. Evidence For**                                                                                            │
│  - **n‑back → 8Ad (anterior)**: The n‑back contrast showed strongest activations at the anterior‑dorsal border  │
│  of area 8Ad, a region classified as ANTERIOR in the Glasser HCP MMP1.0 parcellation. This aligns with the      │
│  claim that an abstract (n‑back) task recruits more anterior PFC.                                               │
│  - **stop → p9‑46v anterior‑dorsal border** (mentioned in Assem 2024): Although stop activations overlapped     │
│  mid‑lateral regions (area 46, 6r), the authors note that “p9‑46v anterior‑dorsal border” was among the         │
│  strongest overlap sites for stop>no‑stop, indicating a putative anterior‑leaning locus even for a task         │
│  classified as abstract.                                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **3. Evidence Against**                                                                                        │
│  - **switch → area 6a (posterior)**: The switch>no‑switch contrast preferentially activated area 6a, a          │
│  POSTERIOR region (mid‑frontal to premotor). This directly opposes the expectation that more abstract tasks     │
│  would drive more anterior PFC activation.                                                                      │
│  - **stop activation pattern**: The dominant overlap for stop>no‑stop was with CON (area 46) and 6r, both       │
│  categorized as MID‑POSTERIOR. Thus, an abstract task (stop) does not preferentially recruit the most anterior  │
│  PFC as the gradient hypothesis predicts.                                                                       │
│                                                                                                                 │
│  ---                                                   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Evaluate whether findings from Ito (2023) and Assem (2024) support:                                      │
│  'More abstract cognitive tasks activate more anterior PFC regions.'                                            │
│                                                                                                                 │
│  IMPORTANT: Use the anatomical reference below to classify any named frontal region as anterior, mid, or        │
│  posterior — do NOT wait for the papers to use the word 'anterior' explicitly.                                  │
│                                                                                                                 │
│                                                                                                                 │
│  FRONTAL LOBE ANTERIOR-POSTERIOR REFERENCE (Glasser HCP MMP1.0 parcellation):                                   │
│                                                                                                                 │
│  ANTERIOR PFC (rostral):                                                                                        │
│  - Area 10 / BA10 / frontopolar cortex — most anterior                                                          │
│  - Area 9 (dorsomedial PFC, 9m)                                                                                 │
│  - Area 8Ad (anterior part of area 8, borders DMN) — anterior                                                   │
│  - Area p9-46v anterior-dorsal border — anterior                                                                │
│  - Area 47 / 47l (lateral orbitofrontal) — anterior-ventral                                                     │
│                                                                                                                 │
│  MID-LATERAL PFC:                                                                                               │
│  - Area 46 / 9-46v / p9-46v — mid-lateral DLPFC                                                                 │
│  - Area 9-46d — mid-lateral DLPFC                                                                               │
│  - Area 8Av / 8C — mid-frontal                                                                                  │
│                                                                                                                 │
│  POSTERIOR/PREMOTOR PFC:                                                                                        │
│  - Area 6 / 6a (dorsal premotor, DAN-associated) — posterior frontal                                            │
│  - Area 6r / IFJ (inferior frontal junction) — posterior-lateral                                                │
│  - Area 55b — posterior lateral PFC                                                                             │
│  - FOP5 (frontal operculum) — posterior-ventral                                                                 │
│                                                                                                                 │
│  KEY MAPPINGS FROM ASSEM (2024):                                                                                │
│  - 3>1 n-back preferring vertices → overlap with 8Ad (ANTERIOR)                                                 │
│  - stop>no stop preferring vertices → overlap with CON, area 46, 6r (MID to POSTERIOR)                          │
│  - switch>no switch preferring vertices → overlap with DAN, area 6a (POSTERIOR)                                 │
│  - p9-46v anterior-dorsal border → stop activations (MI

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 47d2c546-d69a-402f-988a-9d1390126d7f                                                                       │
│  Final Output: **1. Region Classification Table**                                                               │
│                                                                                                                 │
│  | Task | Paper | Region Named | A/P Classification | Abstract? | Gradient Consistent? |                        │
│  |------|-------|--------------|-------------------|-----------|----------------------|                         │
│  | n‑back | Assem 2024 | 8Ad (anterior part of area 8) | ANTERIOR | Yes | **Yes** |                             │
│  | stop | Assem 2024 | CON (area 46, 6r) | MID‑POSTERIOR (i.e., not anterior) | Yes | **No** |                  │
│  | switch | Assem 2024 | DAN / area 6a | POSTERIOR | Yes | **No** |                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **2. Evidence For**                                                                                            │
│  - **n‑back → 8Ad (anterior)**: The n‑back contrast showed strongest activations at the anterior‑dorsal border  │
│  of area 8Ad, a region classified as ANTERIOR in the Glasser HCP MMP1.0 parcellation. This aligns with the      │
│  claim that an abstract (n‑back) task recruits more anterior PFC.                                               │
│  - **stop → p9‑46v anterior‑dorsal border** (mentioned in Assem 2024): Although stop activations overlapped     │
│  mid‑lateral regions (area 46, 6r), the authors note that “p9‑46v anterior‑dorsal border” was among the         │
│  strongest overlap sites for stop>no‑stop, indicating a putative anterior‑leaning locus even for a task         │
│  classified as abstract.                                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **3. Evidence Against**                                                                                        │
│  - **switch → area 6a (posterior)**: The switch>no‑switch contrast preferentially activated area 6a, a          │
│  POSTERIOR region (mid‑frontal to premotor). This directly opposes the expectation that more abstract tasks     │
│  would drive more anterior PFC activation.                                                                      │
│  - **stop activation pattern**: The dominant overlap for stop>no‑stop was with CON (area 46) and 6r, both       │
│  categorized as MID‑POSTERIOR. Thus, an abstract task (stop) does not preferentially recruit the most anterior  │
│  PFC as the gradient hypothesis predicts.                                                                       │
│                                                                                                                 │
│  ---                                                  

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [14]:
# Capture all agent outputs after crew completes
agent_outputs = {
    "classifier": classify_tasks.output.raw,      # Agent 1's markdown table
    "analyst":    analyze_activation.output.raw,  # Agent 2's markdown table  
    "verifier":   verify_claim.output.raw,        # Agent 3's structured verdict
}

# Also capture which chunks were actually retrieved per agent query
# (instrument paper_search to log this)
# retrieval_log = []  # populated by the instrumented tool below

## 8. Results

In [15]:
from IPython.display import Markdown
Markdown(result.raw)

**1. Region Classification Table**

| Task | Paper | Region Named | A/P Classification | Abstract? | Gradient Consistent? |
|------|-------|--------------|-------------------|-----------|----------------------|
| n‑back | Assem 2024 | 8Ad (anterior part of area 8) | ANTERIOR | Yes | **Yes** |
| stop | Assem 2024 | CON (area 46, 6r) | MID‑POSTERIOR (i.e., not anterior) | Yes | **No** |
| switch | Assem 2024 | DAN / area 6a | POSTERIOR | Yes | **No** |

---

**2. Evidence For**  
- **n‑back → 8Ad (anterior)**: The n‑back contrast showed strongest activations at the anterior‑dorsal border of area 8Ad, a region classified as ANTERIOR in the Glasser HCP MMP1.0 parcellation. This aligns with the claim that an abstract (n‑back) task recruits more anterior PFC.  
- **stop → p9‑46v anterior‑dorsal border** (mentioned in Assem 2024): Although stop activations overlapped mid‑lateral regions (area 46, 6r), the authors note that “p9‑46v anterior‑dorsal border” was among the strongest overlap sites for stop>no‑stop, indicating a putative anterior‑leaning locus even for a task classified as abstract.

---

**3. Evidence Against**  
- **switch → area 6a (posterior)**: The switch>no‑switch contrast preferentially activated area 6a, a POSTERIOR region (mid‑frontal to premotor). This directly opposes the expectation that more abstract tasks would drive more anterior PFC activation.  
- **stop activation pattern**: The dominant overlap for stop>no‑stop was with CON (area 46) and 6r, both categorized as MID‑POSTERIOR. Thus, an abstract task (stop) does not preferentially recruit the most anterior PFC as the gradient hypothesis predicts.

---

**4. Data Gaps**  

- **Assem 2024** – *Design gap*: The three executive tasks (n‑back, switch, stop) were not orthogonal manipulations of abstraction level; they were contrasted only against a baseline. Consequently, the experiment was not designed to test a systematic anterior‑posterior gradient of abstraction.  
- **Ito 2023** – *Retrieval gap*: The manuscript classifies tasks as abstract or concrete but provides no activation maps of specific frontal subdivisions (e.g., 8Ad, area 6a). Thus, the data cannot be used to assign any of the reported frontal regions to an anterior‑posterior axis.

---

**5. Verdict**  
The evidence **partially supports** the claim that more abstract cognitive tasks activate more anterior PFC. The n‑back task engages the anterior region 8Ad, consistent with the gradient hypothesis, but the switch and stop tasks activate posterior‑likely regions (area 6a and mid‑posterior CON), contradicting the prediction. Moreover, neither study was explicitly designed to test a systematic anterior‑posterior gradient, leaving the claim under‑determined.

---

**6. Key References**  
- Assem 2024, *Demand‑based executive function topography*, Cerebral Cortex 34(2): 13‑20.  
- Ito 2023, *Hierarchical task classification and competition in cognition*, (preprint).  
- Glasser et al. 2016, *A multi‑modal parcellation of human cortex*, NeuroImage.  

---

In [25]:
import re
import pandas as pd

def parse_markdown_table(markdown_text: str) -> pd.DataFrame:
    # Normalise unicode dashes and whitespace
    text = markdown_text.replace("\u2011", "-").replace("\u2012", "-")
    lines = text.strip().split("\n")
    table_lines = [l for l in lines if l.strip().startswith("|")]
    
    if len(table_lines) < 3:
        print("⚠️  No table found — agent may have returned prose instead of markdown")
        return pd.DataFrame()
    
    headers = [h.strip() for h in table_lines[0].split("|") if h.strip()]
    n_cols = len(headers)
    rows = []
    
    for line in table_lines[2:]:
        # Strip leading/trailing pipes, then split
        stripped = line.strip().strip("|")
        cells = [c.strip() for c in stripped.split("|")]
        
        if len(cells) == 0:
            continue
        elif len(cells) >= n_cols:
            # Merge any overflow cells back into the last column
            merged = cells[:n_cols - 1] + [" | ".join(cells[n_cols - 1:])]
            rows.append(dict(zip(headers, merged)))
        else:
            print(f"⚠️  Skipped row with too few cells ({len(cells)}<{n_cols}): {line[:80]}")
    
    df = pd.DataFrame(rows)
    print(f"✅  Parsed {len(df)} rows, {len(df.columns)} columns")
    return df
  

# Parse each agent's output
classifier_df = parse_markdown_table(agent_outputs["classifier"])
analyst_df    = parse_markdown_table(agent_outputs["analyst"])

print("Classifier table:")
display(classifier_df)
print("\nAnalyst table:")
display(analyst_df)

✅  Parsed 29 rows, 5 columns
✅  Parsed 3 rows, 5 columns
Classifier table:


,Task Name,Paper,Abstract / Concrete,Hierarchy Level,Justification
0,"Go, no-go",Ito (2023),Concrete,RESPONSE,Concrete; competition among manual responses (...
1,Arithmetic,Ito (2023),Abstract,CONTEXT,Abstract; competition among contextual rule se...
2,IAPS aﬀective,Ito (2023),Concrete,RESPONSE,Concrete; passive viewing of affective picture...
3,IAPS emotion,Ito (2023),Concrete,RESPONSE,Concrete; passive viewing of affective picture...
4,Object viewing,Ito (2023),Concrete,RESPONSE,Concrete; passive viewing of objects (no rule ...
5,Interval timing,Ito (2023),Concrete,RESPONSE,Concrete; interval timing tasks involve no rul...
6,Motor imagery,Ito (2023),Concrete,RESPONSE,Concrete; motor imagery involves no overt resp...
7,Stroop,Ito (2023),Abstract,FEATURE,Abstract; competition among perceptual feature...
8,Verbal n-back,Ito (2023),Abstract,DIMENSION,Abstract; competition among perceptual dimensi...
9,Nature movie,Ito (2023),Concrete,RESPONSE,Concrete; passive movie watching (no rule sele...



Analyst table:


,Task,Paper,Frontal Region Named,Anterior or Posterior,Evidence Quote / Figure Reference
0,n-back,Assem (2024),core MD,Posterior,strong activation throughout core MD (Figs. 4 ...
1,switch,Assem (2024),DAN,Posterior,Switch>no switch activations were left lateral...
2,stop,Assem (2024),CON,Anterior,most prominently CON for stop > no stop (t36 =...


## 9. Retrieval Quality Audit

Inspect which chunks were actually retrieved and used — important for debugging RAG hallucinations and validating that the correct papers were consulted.

In [17]:
def audit_retrieval(query: str, top_k: int = 3) -> None:
    """Display the top-k retrieved chunks for a given query to validate retrieval quality."""
    docs = retriever.invoke(query)
    print(f"Query: '{query}'")
    print(f"Retrieved {len(docs)} chunks\n")
    for i, doc in enumerate(docs[:top_k]):
        print(f"--- Chunk {i+1} ---")
        print(f"Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page', '?')}")
        print(doc.page_content[:400])
        print()

# Audit key queries used in the pipeline
audit_retrieval("Badre D'Esposito hierarchical abstraction definition")
audit_retrieval("PFC activation results Ito 2023 frontal cortex")

Query: 'Badre D'Esposito hierarchical abstraction definition'
Retrieved 6 chunks

--- Chunk 1 ---
Source: data/2007_badre.pdf | Page: 2
or superordinate representation comprises a category or
class of subordinate representations. Hence, our con-
cept of abstractness derives from a classing rule, implicit
in the tree structure of a hierarchy. Consider, as an ex-
ample, a hierarchy of semantic representations. The
concept ‘‘dog’’ is more abstract and is superordinate
to the concept ‘‘spaniel’’ because the former is a class
that incl

--- Chunk 2 ---
Source: data/2007_badre.pdf | Page: 0
Braver, & Cohen, 2002; Christoff & Gabrieli, 2000;
D’Esposito, Postle, & Rypma, 2000; Fuster, 1997). The hi-
erarchy hypothesis derives from the central assumption
that the frontal lobes are critical for the selection and
execution of action (Fuster, 1997). Broadly construed,
this function entails specifying an abstract action goal, like
driving to work, down to a concrete instantiation in the
for

--- Ch

## 10. Discussion & Limitations

### What this pipeline does well
- Grounds LLM reasoning in retrieved text via explicit source citations
- Separates concerns across three agents with distinct epistemic roles
- The verifier is designed to flag unsupported inferences, not just confirm them

### Known limitations

| Limitation | Impact | Mitigation |
|---|---|---|
| MDTB (Ito 2023) reports whole-cortex topography, not per-task PFC subregion data | Agent 2 may have limited data for specific tasks | Add a targeted search for supplementary figures |
| bioRxiv scraping may fail silently in restricted network environments | Agent 3 may miss recent literature | Fall back to PubMed API or Semantic Scholar |
| Free-tier LLMs have variable instruction-following | Structured table outputs may be malformed | Add output parsers / Pydantic schemas |
| No re-ranking of retrieved chunks | Less relevant chunks may crowd out key passages | Add a cross-encoder reranker (e.g., `ms-marco-MiniLM`) |

### Potential extensions
- **Reranking:** Add a cross-encoder reranker between retrieval and generation
- **Structured outputs:** Use Pydantic models as `expected_output` schemas for parseable results
- **Citation tracing:** Log which chunks each agent used for full provenance
- **HyDE (Hypothetical Document Embeddings):** Generate a hypothetical answer first, embed it, then retrieve — improves recall for scientific queries

In [18]:
# ── Section 11: Systematic Evaluation ────────────────────────────────────────
import json

# ── 11a. Ground Truth (build once, reuse across runs) ─────────────────────────

gold_classifications = {
    ("n-back",       "Assem"): ("Abstract",  "dimension"),
    ("switch",       "Assem"): ("Abstract",  "context"),
    ("stop signal",  "Assem"): ("Concrete",  "response"),
    ("CPRO",         "Ito"):   ("Abstract",  "context"),
    ("Stroop",       "Ito"):   ("Abstract",  "feature"),
    ("Verbal n-back","Ito"):   ("Abstract",  "dimension"),
    ("Go, no-go",    "Ito"):   ("Concrete",  "response"),
    ("Object n-back","Ito"):   ("Abstract",  "dimension"),
    ("Visual search","Ito"):   ("Abstract",  "feature"),
    ("Verb generation","Ito"): ("Abstract",  "context"),
    # extend for all 29 tasks...
}

In [19]:
gold_activations = {
    ("n-back",  "Assem"): {"region": "8Ad",     "ap": "ANTERIOR"},
    ("switch",  "Assem"): {"region": "6a",       "ap": "POSTERIOR"},
    ("stop",    "Assem"): {"region": "area 46",  "ap": "MID"},
    ("CPRO",    "Ito"):   {"region": "DESIGN_GAP", "ap": None},
    ("Stroop",  "Ito"):   {"region": "DESIGN_GAP", "ap": None},
}

retrieval_eval_set = [
    {
        "query":            "Badre D'Esposito hierarchical abstraction definition",
        "relevant_sources": ["data/2007_badre.pdf"],
        "relevant_pages":   [2, 0],
    },
    {
        "query":            "Assem 2024 n-back dorsal frontal 8Ad anterior DMN",
        "relevant_sources": ["data/2024_demand.pdf"],
        "relevant_pages":   [8, 15],
    },
    {
        "query":            "MDTB 26 tasks list multitask fMRI Ito 2023",
        "relevant_sources": ["data/2023_multitask.pdf"],
        "relevant_pages":   [2, 10],
    },
    {
        "query":            "Assem 2024 stop CON switch DAN frontal preference vertices",
        "relevant_sources": ["data/2024_demand.pdf"],
        "relevant_pages":   [9, 6],
    },
]

In [34]:
# ── 11b. Layer 1: Retriever Evaluation ────────────────────────────────────────

def evaluate_retriever(eval_set, retriever, k=6):
    records = []
    for item in eval_set:
        docs = retriever.invoke(item["query"])
        retrieved = [
            (doc.metadata.get("source"), doc.metadata.get("page"))
            for doc in docs
        ]

        # Build the ground-truth set of (source, page) pairs
        relevant_pairs = {
            (src, pg)
            for src in item["relevant_sources"]
            for pg in item["relevant_pages"]
        }

        # Deduplicate retrieved pairs before counting
        retrieved_pairs = set(retrieved)

        relevant_retrieved = len(retrieved_pairs & relevant_pairs)  # set intersection
        total_relevant     = len(relevant_pairs)

        precision = relevant_retrieved / len(retrieved_pairs) if retrieved_pairs else 0
        recall    = relevant_retrieved / total_relevant       if total_relevant  else 0

        records.append({
            "query":             item["query"][:60],
            "context_precision": round(precision, 2),
            "context_recall":    round(recall,    2),
        })

    df = pd.DataFrame(records)
    mean_row = df[["context_precision", "context_recall"]].mean().to_frame().T
    mean_row.index = ["mean"]
    return pd.concat([df, mean_row])

retriever_scores = evaluate_retriever(retrieval_eval_set, retriever)
print("── Retriever Scores ──")
display(retriever_scores)

── Retriever Scores ──


,query,context_precision,context_recall
0,Badre D'Esposito hierarchical abstraction defi...,0.400,1.000
1,Assem 2024 n-back dorsal frontal 8Ad anterior DMN,0.200,0.500
2,MDTB 26 tasks list multitask fMRI Ito 2023,0.670,1.000
3,Assem 2024 stop CON switch DAN frontal prefere...,0.670,1.000
mean,NaN,0.485,0.875


In [35]:
# ── 11c. Layer 2: Classifier Evaluation ───────────────────────────────────────

def evaluate_classifier(classifier_df, gold):
    correct_class = correct_level = has_justification = matched = 0
    total = len(gold)

    for (task, paper_key), (gold_class, gold_level) in gold.items():
        # flexible match: "Assem" matches "Assem (2024)", etc.
        mask = (
            classifier_df.iloc[:, 0].str.contains(task, case=False, na=False) &
            classifier_df.iloc[:, 1].str.contains(paper_key, case=False, na=False)
        )
        row = classifier_df[mask]
        if row.empty:
            continue
        matched += 1

        pred_class = row.iloc[0, 2]   # "Abstract / Concrete" column
        pred_level = row.iloc[0, 4]   # "Hierarchy Level" column (index may vary)
        justif     = row.iloc[0, -1]  # last column = justification

        if gold_class.lower() in pred_class.lower():
            correct_class += 1
        if gold_level.lower() in pred_level.lower():
            correct_level += 1
        if len(justif) > 30:
            has_justification += 1

    return {
        "tasks_matched":          matched,
        "classification_accuracy": round(correct_class / total, 2),
        "level_accuracy":          round(correct_level / total, 2),
        "justification_coverage":  round(has_justification / total, 2),
    }

classifier_scores = evaluate_classifier(classifier_df, gold_classifications)
print("\n── Classifier Scores ──")
display(pd.DataFrame([classifier_scores]))


── Classifier Scores ──


,tasks_matched,classification_accuracy,level_accuracy,justification_coverage
0,10,0.9,0.9,1.0


In [36]:
# ── 11d. Layer 3: Analyst Evaluation ──────────────────────────────────────────

def evaluate_analyst(analyst_df, gold):
    region_correct = ap_correct = design_gap_correct = matched = 0
    total = len(gold)

    for (task, paper_key), gold_vals in gold.items():
        mask = analyst_df.iloc[:, 0].str.contains(task, case=False, na=False)
        row  = analyst_df[mask]
        if row.empty:
            continue
        matched += 1

        pred_region = str(row.iloc[0, 2])
        pred_ap     = str(row.iloc[0, 3])

        if gold_vals["region"] == "DESIGN_GAP":
            if any(kw in pred_region.upper() for kw in ["GAP", "N/A", "NOT", "NONE", "DESIGN"]):
                design_gap_correct += 1
        else:
            if gold_vals["region"].lower() in pred_region.lower():
                region_correct += 1
            if gold_vals["ap"] and gold_vals["ap"] in pred_ap.upper():
                ap_correct += 1

    design_gap_total = sum(1 for v in gold.values() if v["region"] == "DESIGN_GAP")
    factual_total    = total - design_gap_total

    return {
        "tasks_matched":       matched,
        "task_coverage":       round(matched / total, 2),          # ← new
        "region_accuracy":     round(region_correct / factual_total, 2) if factual_total else None,
        "ap_accuracy":         round(ap_correct     / factual_total, 2) if factual_total else None,
        "design_gap_recall":   round(design_gap_correct / design_gap_total, 2) if design_gap_total else None,
    }

analyst_scores = evaluate_analyst(analyst_df, gold_activations)
print("\n── Analyst Scores ──")
display(pd.DataFrame([analyst_scores]))


── Analyst Scores ──


,tasks_matched,task_coverage,region_accuracy,ap_accuracy,design_gap_recall
0,3,0.6,0.0,0.33,0.0


In [28]:
# ── 11e. Layer 4: Verifier Evaluation (LLM-as-judge) ─────────────────────────

VERIFIER_JUDGE_PROMPT = """
You are a scientific peer reviewer evaluating an AI agent's verdict on a neuroscience claim.

Research question: Do more abstract cognitive tasks preferentially activate more anterior PFC?

Agent's full output:
{verdict_text}

Ground truth context:
- n-back (abstract) → 8Ad (ANTERIOR) [Assem 2024]: supports claim
- switch (abstract) → 6a (POSTERIOR) [Assem 2024]: contradicts claim
- stop (abstract per agent, concrete per gold) → area 46 [Assem 2024]: contradicts claim
- Ito (2023) has NO explicit frontal subregion data for any task → design gap
- Correct verdict: "Partially Supported"

Score each dimension 0-2:
- faithfulness: are region-task pairs grounded in retrieved text without fabrication?
  (0=fabricated regions, 1=partially grounded, 2=all claims traceable to retrieved text)
- calibration: is the verdict appropriate given the evidence?
  (0=wrong verdict, 1=correct but poorly justified, 2=correct with specific citations)
- gap_noted: does the agent flag that neither Ito 2023 nor Assem 2024 was designed to test this?
  (0=not mentioned, 1=mentioned for one paper, 2=mentioned for both papers)
- coverage: does the agent report findings from BOTH Ito (2023) AND Assem (2024)?
  (0=only one paper cited, 1=both mentioned but one superficially, 2=both substantively used)

Respond ONLY with valid JSON, no preamble:
{{"faithfulness": 0-2, "calibration": 0-2, "gap_noted": 0-2, "coverage": 0-2, "rationale": "two sentences"}}
"""

FAITHFULNESS_PROMPT = """
You are evaluating whether a task classification is grounded in the cited text.

Task: {task}
Classification: {classification}
Hierarchy Level: {level}
Agent justification: {justification}

The correct definitions from Badre & D'Esposito (2007):
- Response level: competition among manual responses
- Feature level: competition among perceptual feature-to-response mappings
- Dimension level: competition among perceptual dimensions
- Context level: competition among sets of contextual associations

Is the classification and level consistent with these definitions?
Respond ONLY with valid JSON:
{{"score": 0_or_1, "reason": "one sentence"}}
"""

In [29]:
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)

def llm_judge(prompt: str) -> dict:
    response = judge_llm.invoke(prompt)
    try:
        return json.loads(response.content)
    except json.JSONDecodeError:
        return {"error": "parse_failed", "raw": response.content}

In [30]:
def run_llm_judge(prompt: str) -> dict:
    response = judge_llm.invoke(prompt)
    try:
        # strip markdown fences if present
        text = re.sub(r"```json|```", "", response.content).strip()
        return json.loads(text)
    except json.JSONDecodeError:
        return {"error": "parse_failed", "raw": response.content[:200]}

# Verifier: single LLM-as-judge call
verifier_prompt = VERIFIER_JUDGE_PROMPT.format(verdict_text=agent_outputs["verifier"])
verifier_scores = run_llm_judge(verifier_prompt)
print("\n── Verifier LLM-as-Judge Scores ──")
display(pd.DataFrame([verifier_scores]))


── Verifier LLM-as-Judge Scores ──


,faithfulness,calibration,gap_noted,coverage,rationale
0,2,2,2,2,The agent correctly maps all task‑region pairs...


In [37]:
# Classifier faithfulness: sample rows and judge each
faithfulness_scores = []
sample_rows = classifier_df.sample(min(5, len(classifier_df)), random_state=42)
for _, row in sample_rows.iterrows():
    cols = classifier_df.columns.tolist()
    prompt = FAITHFULNESS_PROMPT.format(
        task           = row[cols[0]],
        classification = row[cols[2]],
        level          = row[cols[3]] if len(cols) > 3 else "unknown",
        justification  = row[cols[-1]],
    )
    result_j = run_llm_judge(prompt)
    faithfulness_scores.append(result_j.get("score", 0))

print(f"\n── Classifier Faithfulness (sampled {len(faithfulness_scores)} rows) ──")
print(f"Score: {sum(faithfulness_scores)}/{len(faithfulness_scores)}")


── Classifier Faithfulness (sampled 5 rows) ──
Score: 2/5


In [38]:
# ── 11f. Summary Report ───────────────────────────────────────────────────────

summary = {
    "retriever_precision_mean":   retriever_scores.loc["mean", "context_precision"],
    "retriever_recall_mean":      retriever_scores.loc["mean", "context_recall"],
    "classifier_accuracy":        classifier_scores["classification_accuracy"],
    "classifier_level_accuracy":  classifier_scores["level_accuracy"],
    "classifier_faithfulness":    round(sum(faithfulness_scores) / len(faithfulness_scores), 2),
    "analyst_region_accuracy":    analyst_scores["region_accuracy"],
    "analyst_ap_accuracy":        analyst_scores["ap_accuracy"],
    "analyst_gap_recall":         analyst_scores["design_gap_recall"],
    "verifier_faithfulness":      verifier_scores.get("faithfulness"),
    "verifier_calibration":       verifier_scores.get("calibration"),
    "verifier_gap_noted":         verifier_scores.get("gap_noted"),
    "verifier_attribution":       verifier_scores.get("attribution"),
}

print("\n══ Final Evaluation Summary ══")
display(pd.DataFrame([summary]).T.rename(columns={0: "score"}))


══ Final Evaluation Summary ══


,score
retriever_precision_mean,0.485
retriever_recall_mean,0.875
classifier_accuracy,0.9
classifier_level_accuracy,0.9
classifier_faithfulness,0.4
analyst_region_accuracy,0.0
analyst_ap_accuracy,0.33
analyst_gap_recall,0.0
verifier_faithfulness,2
verifier_calibration,2


In [ ]:
# import json
# from langchain_openai import ChatOpenAI

# judge_llm = ChatOpenAI(
#     model="nvidia/nemotron-3-nano-30b-a3b:free",
#     openai_api_key=OPENROUTER_API_KEY,
#     openai_api_base="https://openrouter.ai/api/v1",
# )

# def llm_judge(prompt: str) -> dict:
#     response = judge_llm.invoke(prompt)
#     try:
#         return json.loads(response.content)
#     except json.JSONDecodeError:
#         return {"error": "parse_failed", "raw": response.content}

# def run_full_evaluation(crew_result, retriever, agent_outputs: dict):
#     report = {}

#     # Layer 1: Retriever
#     report["retriever"] = evaluate_retriever(retrieval_eval, retriever)

#     # Layer 2: Classifier
#     classifier_df  = parse_markdown_table(agent_outputs["classifier"])
#     report["classifier_structural"] = evaluate_classifier_output(classifier_df, gold_classifications)

#     # Faithfulness of justifications (sample 5 rows)
#     faithfulness_scores = []
#     for _, row in classifier_df.sample(5).iterrows():
#         prompt = faithfulness_prompt.format(**row)
#         faithfulness_scores.append(llm_judge(prompt)["score"])
#     report["classifier_faithfulness"] = sum(faithfulness_scores) / len(faithfulness_scores)

#     # Layer 3: Analyst
#     analyst_df = parse_markdown_table(agent_outputs["analyst"])
#     report["analyst_structural"] = evaluate_analyst_output(analyst_df, gold_activations)

#     # Layer 4: Verifier
#     report["verifier_structural"] = evaluate_verifier_structure(crew_result.raw)
#     verdict_prompt = verifier_judge_prompt.format(verdict_text=crew_result.raw)
#     report["verifier_llm_judge"]   = llm_judge(verdict_prompt)

#     return report

# results = run_full_evaluation(result, retriever, agent_outputs)